# Zolai LLM Fine-Tuning on T4x2

Training Qwen 7B on 6.4M Zolai language examples (500K samples in 30h)

## 1. Setup

In [ ]:
# Install dependencies
!pip install -q transformers peft datasets accelerate

# Check GPU
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}GB")
print(f"GPU Count: {torch.cuda.device_count()}")

## 2. Load Dataset

In [ ]:
import json
from pathlib import Path
from datasets import Dataset

# Load from Kaggle input
TRAIN_FILE = Path("/kaggle/input/zolai-dataset/llm_train.jsonl")
VAL_FILE = Path("/kaggle/input/zolai-dataset/llm_val.jsonl")

def load_dataset(filepath, max_samples=500000):
    texts = []
    with open(filepath, "r") as f:
        for i, line in enumerate(f):
            if i >= max_samples:
                break
            rec = json.loads(line)
            texts.append(rec.get("text", ""))
    return Dataset.from_dict({"text": texts})

print("Loading datasets...")
train_dataset = load_dataset(TRAIN_FILE, max_samples=500000)
val_dataset = load_dataset(VAL_FILE, max_samples=50000)

print(f"Train: {len(train_dataset):,} samples")
print(f"Val: {len(val_dataset):,} samples")

## 3. Load Model

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import get_peft_model, LoraConfig, TaskType

MODEL_NAME = "Qwen/Qwen-7B-Chat"

print(f"Loading model: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

# LoRA config
lora_config = LoraConfig(
    r=4,
    lora_alpha=8,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 4. Tokenize

In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=256
    )

print("Tokenizing...")
train_dataset = train_dataset.map(tokenize_function, batched=True, batch_size=1000)
val_dataset = val_dataset.map(tokenize_function, batched=True, batch_size=1000)

print("✓ Tokenization complete")

## 5. Train

In [ ]:
from transformers import TrainingArguments, Trainer

OUTPUT_DIR = "/kaggle/working/zolai_llm_t4x2"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_steps=100,
    weight_decay=0.01,
    logging_steps=50,
    save_steps=200,
    eval_steps=200,
    evaluation_strategy="steps",
    save_strategy="steps",
    load_best_model_at_end=True,
    fp16=True,
    max_steps=2000,
    dataloader_num_workers=2,
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
)

print("\n" + "="*80)
print("STARTING TRAINING (30h limit)")
print("="*80 + "\n")

trainer.train()

## 6. Save Model

In [ ]:
print("Saving model...")
model.save_pretrained(f"{OUTPUT_DIR}/final_model")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/final_model")

print(f"\n✅ Training complete!")
print(f"Model saved to: {OUTPUT_DIR}/final_model")

## 7. Download Results

In [ ]:
import os
import shutil

# Compress model
print("Compressing model...")
shutil.make_archive(
    "/kaggle/working/zolai_model",
    "gztar",
    f"{OUTPUT_DIR}",
    "final_model"
)

print("✅ Model compressed: /kaggle/working/zolai_model.tar.gz")
print(f"Size: {os.path.getsize('/kaggle/working/zolai_model.tar.gz') / 1e9:.1f}GB")

## 8. Evaluate on Test Set

In [ ]:
# Load test set
TEST_FILE = Path("/kaggle/input/zolai-dataset/llm_test.jsonl")
test_dataset = load_dataset(TEST_FILE, max_samples=10000)
test_dataset = test_dataset.map(tokenize_function, batched=True, batch_size=1000)

# Evaluate
print("Evaluating on test set...")
test_results = trainer.evaluate(eval_dataset=test_dataset)

print("\nTest Results:")
for key, value in test_results.items():
    print(f"  {key}: {value:.4f}")